# 音乐源分离 - pymss WebUI

This notebook installs [pymss](https://github.com/pymss-project/pymss) (Python Music Source Separation) on Google Colab with GPU, starts the WebUI server, and exposes it via a **public temporary URL** using Cloudflare Quick Tunnel.

**What you get:** A browser-accessible WebUI where you can upload audio and separate vocals, drums, bass, and other instruments.

**Requirements:** Colab runtime with GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
# ============================================================
# pymss Colab - One-click: install + server + public URL
# ============================================================

import torch, subprocess, time, re, urllib.request, shutil, os, threading, tempfile

PORT = 8000
SERVER_LOG = "/tmp/pymss_server.log"

# ---- 1. Check GPU ----
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[GPU] {gpu_name} ({gpu_mem:.1f} GB)")

# ---- 2. Install pymss[server] ----
!pip install -q "pymss[server]"
pymss_ver = subprocess.check_output(["pip", "show", "pymss"], text=True)
pymss_ver = [l.split(":", 1)[1].strip() for l in pymss_ver.splitlines() if l.startswith("Version:")][0]
print(f"[pymss] {pymss_ver}")

# ---- 3. Download cloudflared ----
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
cf_ver = subprocess.check_output(["cloudflared", "--version"], text=True).strip()
print(f"[cloudflared] {cf_ver}")

# ---- 4. Start pymss server with log streaming ----
print("\n[pymss] Starting server (model loads on first request)...")
log_file = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(
    ["pymss", "serve", "--debug",
     "--host", "0.0.0.0", "--port", str(PORT),
     "--device", "auto", "--source", "modelscope", "--webui"],
    stdout=log_file, stderr=subprocess.STDOUT,
)

def stream_server_log():
    """Read server log file and print new lines to cell output."""
    last_pos = 0
    while server_proc.poll() is None:
        try:
            with open(SERVER_LOG, "r") as f:
                f.seek(last_pos)
                new = f.read()
                if new:
                    print(new, end="", flush=True)
                    last_pos = f.tell()
        except FileNotFoundError:
            pass
        time.sleep(0.5)

log_thread = threading.Thread(target=stream_server_log, daemon=True)
log_thread.start()

for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        print(f"[pymss] Server ready on port {PORT}.")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server failed to start within 120s.")

# ---- 5. Launch public tunnel (output to temp file, not terminal) ----
tunnel_log = tempfile.NamedTemporaryFile(mode="w", suffix=".log", delete=False)
print("[tunnel] Starting cloudflared...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
url_re = re.compile(r"(https://[a-z0-9\-]+\.trycloudflare\.com)")
public_url = None
start = time.time()
while time.time() - start < 30:
    try:
        with open(tunnel_log.name) as f:
            content = f.read()
        m = url_re.search(content)
        if m:
            public_url = m.group(1)
            break
    except FileNotFoundError:
        pass
    time.sleep(0.5)
os.unlink(tunnel_log.name)
if not public_url:
    raise RuntimeError("Tunnel failed to start within 30s.")

print(f"\n{'='*60}")
print(f"  Public URL:  {public_url}")
print(f"  WebUI:       {public_url}/ui/")
print(f"  API docs:    {public_url}/docs")
print(f"{'='*60}\n")
print("Open the WebUI link above to start separating audio.")
print("Stop: tunnel_proc.terminate() + server_proc.terminate()")

---

### Quick Reference

**WebUI:** Open the URL above → browse catalog → load model → upload audio → separate.

**Python API:**
```python
from pymss import MSSeparator
with MSSeparator.from_model_name("bs_roformer_voc_hyperacev2", download=True, device="auto", output_format="wav", store_dirs="results") as sep:
    sep.process_folder("/path/to/audio")
```

**Cleanup:**
```python
tunnel_proc.terminate()
server_proc.terminate()
```